In [35]:
import MDAnalysis as mda
from openmm.app import AmberPrmtopFile, PDBFile

In [36]:
sys_name='6e23_A'
prmtop = '../equilibration/6e23_A/system.prmtop'
pdb = '../equilibration/6e23_A/system.pdb'
pdb_fname = f'../equilibration/{sys_name}/equilibration/{sys_name}_equilibrated.pdb'
traj = '../equilibration/6e23_A/equilibration/equilibration_6e23_A_aligned.dcd'
u = mda.Universe(prmtop, traj)
# atoms = [atom.index for atom in u.select_atoms('resid 218') if atom.name == 'CA']
# atoms.resname

In [37]:
# topology = AmberPrmtopFile(prmtop).topology
topology = PDBFile(pdb_fname).topology

In [38]:
pocket_residues = [218, 219, 262, 263, 305, 306, 49, 50, 91, 92, 133, 134, 175, 176]  # Change this to your pocket residue IDs
pocket_residues = [str(res) for res in pocket_residues]  # Convert to strings if necessary
pocket_CA_ids = []
for residue in topology.residues():
    if residue.id in pocket_residues and residue.name not in ['HOH', 'WAT','UNK']:
        print(f'Matching residue: {residue.name} {residue.id}')
        for at in residue.atoms():
            if at.name == 'CA':
                # print(at)
                pocket_CA_ids.append(at.index)
print(f"Number of pocket heavy atoms: {len(pocket_CA_ids)}")
print(pocket_CA_ids)
print(len(pocket_residues))

Matching residue: SER 49
Matching residue: SER 50
Matching residue: SER 91
Matching residue: ASP 92
Matching residue: PHE 133
Matching residue: CYS 134
Matching residue: SER 175
Matching residue: ALA 176
Matching residue: SER 218
Matching residue: PHE 219
Matching residue: ILE 262
Matching residue: PHE 263
Matching residue: ILE 305
Matching residue: SER 306
Number of pocket heavy atoms: 14
[275, 286, 920, 931, 1554, 1574, 2196, 2207, 2831, 2842, 3549, 3568, 4230, 4249]
14


In [39]:
import os
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import MDAnalysis as mda
from MDAnalysis.analysis import align, density

def make_unbinding_paths_pml(
    protein_pdb: str,
    paths: Dict[str, List[Tuple[str, str]]],
    outdir: str = "unbinding_paths_vis",
    ligand_resname: str = "UNK",              # <-- explicit ligand, default 'UNK'
    align_sel: str = "protein and backbone",
    grid_spacing: float = 0.5,                # Å
    density_sigma: float = 1.0,               # Å
    level: float = 0.003,                     # isosurface level
    path_colors: Optional[Dict[str, str]] = None,
    surface_transparency: float = 0.35,
    cartoon_transparency: float = 0.25,
    cartoon_color: str = "palecyan",
    sample_stride: int = 5,
) -> str:
    """
    Build ligand-density maps per unbinding path and write a PyMOL .pml script.

    Parameters
    ----------
    protein_pdb : str
        Reference protein PDB (also used for alignment).
    paths : Dict[str, List[Tuple[topology, trajectory]]]
        Map path name -> list of (AMBER prmtop or equivalent, trajectory file).
    ligand_resname : str
        Residue name of the ligand (e.g., 'LIG'). Default 'UNK'.
    """
    Path(outdir).mkdir(parents=True, exist_ok=True)
    protein_pdb = str(Path(protein_pdb).resolve())
    pml_path = str(Path(outdir, "unbinding_paths.pml").resolve())

    # reference for alignment
    u_ref = mda.Universe(protein_pdb)
    ligand_sel = f"resname {ligand_resname}"

    # default color palette if not provided
    default_palette = ["deepsalmon", "marine", "forest", "violetpurple", "gold", "tv_red", "tv_blue"]
    if path_colors is None:
        path_colors = {name: default_palette[i % len(default_palette)] for i, name in enumerate(paths)}

    dx_files = {}
    for path_name, traj_list in paths.items():
        dens_sum = None
        total_frames = 0

        for top, traj in traj_list:
            u = mda.Universe(top, traj)
            align.AlignTraj(u, u_ref, select=align_sel, in_memory=True).run()

            lig = u.select_atoms(ligand_sel)
            if lig.n_atoms == 0:
                raise ValueError(f"No atoms found for ligand selection '{ligand_sel}' in {traj}.")

            da = density.DensityAnalysis(lig, 
                                         delta=grid_spacing, 
                                        #  sigma=density_sigma, 
                                         padding=1.0)
            da.run(step=sample_stride)
            rho = da.results.density

            if dens_sum is None:
                dens_sum = rho
            else:
                dens_sum.grid += rho.grid

            total_frames += len(u.trajectory[::sample_stride])

        if total_frames > 0:
            dens_sum.grid /= float(total_frames)

        dx_path = Path(outdir, f"{path_name}_ligand_density.dx")
        dens_sum.export(str(dx_path))
        dx_files[path_name] = str(dx_path.resolve())

    # write PyMOL script
    with open(pml_path, "w") as pml:
        pml.write("# PyMOL visualization for ligand unbinding paths\n")
        pml.write("reinitialize\n")
        pml.write("bg_color white\n")
        pml.write("set antialias, 2\n")
        pml.write("set specular, 0.2\n")
        pml.write("set ray_shadow, off\n")
        pml.write(f"set cartoon_transparency, {cartoon_transparency:.2f}\n")
        pml.write(f"load {protein_pdb}, prot\n")
        pml.write("hide everything, prot\n")
        pml.write("show cartoon, prot\n")
        pml.write(f"color {cartoon_color}, prot\n")

        for path_name, dx in dx_files.items():
            map_obj = f"map_{path_name}"
            surf_obj = f"surf_{path_name}"
            col = path_colors[path_name]
            pml.write(f"load {dx}, {map_obj}\n")
            pml.write(f"isosurface {surf_obj}, {map_obj}, {level:.5f}\n")
            pml.write(f"color {col}, {surf_obj}\n")
            pml.write(f"set transparency, {surface_transparency:.2f}, {surf_obj}\n")
            pml.write(f"set two_sided_lighting, on, {surf_obj}\n")

        # optional: show one ligand from the PDB if present
        pml.write(f"select lig_ref, ({ligand_sel}) and prot\n")
        pml.write("if sele count lig_ref > 0:\n")
        pml.write("    create lig, lig_ref\n")
        pml.write("    show spheres, lig\n")
        pml.write("    color lightpink, lig\n")
        pml.write("    set sphere_transparency, 0.35, lig\n")
        pml.write("orient prot\n")
        pml.write(f"png {Path(outdir,'preview.png')}, ray=1, dpi=300\n")

    return pml_path


In [43]:
from pathlib import Path
from typing import Dict, List, Tuple, Optional
import shutil
import MDAnalysis as mda
from MDAnalysis.analysis import align, density

def make_unbinding_paths_pml(
    protein_pdb: str,
    paths: Dict[str, List[Tuple[str, str]]],
    outdir: str = "unbinding_paths_vis",
    ligand_resname: str = "UNK",
    align_sel: str = "protein and backbone",
    grid_spacing: float = 1.0,
    density_sigma: float = 1.0,
    level: float = 0.003,
    path_colors: Optional[Dict[str, str]] = None,
    surface_transparency: float = 0.35,
    cartoon_transparency: float = 0.25,
    cartoon_color: str = "palecyan",
    sample_stride: int = 5,
    copy_protein_into_outdir: bool = True,   # <-- key
) -> str:
    """Generate ligand-path density maps and a PyMOL .pml that uses only relative paths."""
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)

    # Copy the reference PDB into OUTDIR so the .pml can @run anywhere
    if copy_protein_into_outdir:
        prot_copy = outdir / Path(protein_pdb).name
        if Path(protein_pdb).resolve() != prot_copy.resolve():
            shutil.copy2(protein_pdb, prot_copy)
        protein_for_pymol = prot_copy.name        # <-- filename only, relative
    else:
        # still write a relative path if possible
        protein_for_pymol = Path(protein_pdb).name if Path(protein_pdb).parent == Path.cwd() else str(Path(protein_pdb))

    # Use the absolute path ONLY for MDAnalysis work (not saved in .pml)
    protein_abs = str(Path(protein_pdb).resolve())
    u_ref = mda.Universe(protein_abs)
    ligand_sel = f"resname {ligand_resname}"

    default_palette = ["deepsalmon", "marine", "forest", "violetpurple", "gold", "tv_red", "tv_blue"]
    if path_colors is None:
        path_colors = {name: default_palette[i % len(default_palette)] for i, name in enumerate(paths)}

    dx_files_rel = {}  # path -> relative filename for PyMOL
    for path_name, traj_list in paths.items():
        dens_sum = None
        total_frames = 0

        for top, traj in traj_list:
            u = mda.Universe(top, traj)
            align.AlignTraj(u, u_ref, select=align_sel, in_memory=True).run()

            lig = u.select_atoms(ligand_sel)
            if lig.n_atoms == 0:
                raise ValueError(f"No atoms found for '{ligand_sel}' in {traj}.")

            da = density.DensityAnalysis(lig, delta=grid_spacing, 
                                        #  sigma=density_sigma, 
                                         padding=10.0)
            da.run(step=sample_stride)
            rho = da.results.density

            dens_sum = rho if dens_sum is None else dens_sum._replace(grid=dens_sum.grid + rho.grid) or dens_sum
            total_frames += len(u.trajectory[::sample_stride])

        # Normalize and write DX **inside outdir**
        if total_frames > 0:
            dens_sum.grid /= float(total_frames)
        dx_path = outdir / f"{path_name}_ligand_density.dx"
        dens_sum.export(str(dx_path))
        dx_files_rel[path_name] = dx_path.name     # <-- filename only

    # Write the .pml using ONLY filenames (relative to outdir)
    pml_path = outdir / "unbinding_paths.pml"
    with open(pml_path, "w") as pml:
        pml.write("# Relative-path PyMOL visualization for ligand unbinding paths\n")
        pml.write("reinitialize\n")
        pml.write("bg_color white\n")
        pml.write("set ray_opaque_background, 0\n")
        pml.write("set antialias, 2\n")
        pml.write("set specular, 0.2\n")
        pml.write("set ray_shadow, off\n")
        pml.write(f"set cartoon_transparency, {cartoon_transparency:.2f}\n")
        pml.write(f"load {protein_for_pymol}, prot\n")
        pml.write("hide everything, prot\n")
        pml.write("show cartoon, prot\n")
        pml.write(f"color {cartoon_color}, prot\n")

        for path_name, dx_filename in dx_files_rel.items():
            map_obj = f"map_{path_name}"
            surf_obj = f"surf_{path_name}"
            col = path_colors[path_name]
            pml.write(f"load {dx_filename}, {map_obj}\n")
            pml.write(f"isosurface {surf_obj}, {map_obj}, {level:.5f}\n")
            pml.write(f"color {col}, {surf_obj}\n")
            pml.write(f"set transparency, {surface_transparency:.2f}, {surf_obj}\n")
            pml.write(f"set two_sided_lighting, on, {surf_obj}\n")

        pml.write(f"select lig_ref, ({ligand_sel}) and prot\n")
        pml.write("if sele count lig_ref > 0:\n")
        pml.write("    create lig, lig_ref\n")
        pml.write("    show spheres, lig\n")
        pml.write("    color lightpink, lig\n")
        pml.write("    set sphere_transparency, 0.35, lig\n")
        pml.write("orient prot\n")
        pml.write("zoom prot, 2.0\n")
        pml.write("png preview.png, ray=1, dpi=300\n")

    return str(pml_path)


In [44]:
sys_name='6e23_A'
prmtop = '../equilibration/6e23_A/system.prmtop'
path_A = '6e23_A/sMD/sMD_traj_replica-1_v0.001_forward_aligned.dcd'
path_B = '6e23_A/sMD/sMD_traj_replica-2_v0.001_forward_aligned.dcd'
pdb_fname = f'../equilibration/{sys_name}/equilibration/{sys_name}_equilibrated.pdb'

paths = {
    "path_A": [(prmtop, path_A)],
    "path_B": [(prmtop, path_B)]
}


pml = make_unbinding_paths_pml(
    protein_pdb=pdb_fname,
    paths=paths,
    ligand_resname="UNK",      # or leave as default 'UNK'
    level=0.004,
    path_colors={"path_A":"deepsalmon","path_B":"marine"},
)
# Then: @unbinding_paths_vis/unbinding_paths.pml in PyMOL
